# Bayesovský AR model: předpověď dalších deseti minut pomocí MCMC

Spusťte buňky shora dolů. Připravte alespoň čtyři dostupné dny minutových CSV
se sloupci `Timestamp`, `Čas`, `ODCHYLKA ČR [MW]` z období bez změny letního a
zimního času. CSV se nahrávají do vašeho prostředí Colabu, nikoli na GitHub.

Výchozí AR(5) používá pět posledních známých hodnot. Parametry se učí z minulých
úplných čtvrthodin Gibbsovým MCMC vzorkováním. Každý testovací den používá jen
starší dny. Například z `12:00–12:04` předpovídá `12:05–12:14`.

Výstup obsahuje pravděpodobnosti znaménka, 90% predikční intervaly a srovnání
s jednoduchými metodami. Dobrá diagnostika MCMC sama o sobě neověřuje přesnost
předpovědi; několik testovacích dnů nestačí k dlouhodobému závěru.

[Podrobný návod](https://github.com/000307/Odchylky/blob/main/BAYES_AR_MCMC.md)


## 1. Nainstalovat závislosti

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "numpy", "pandas", "scipy", "matplotlib", "scikit-learn", "arviz>=0.20,<0.23"],
    check=True,
)


## 2. Stáhnout potřebné skripty z GitHubu

Stahují se tři soubory: hlavní skript, jádro MCMC a společné načítání CSV.
Pro opakování přesné verze můžete `REF` změnit z `main` na konkrétní SHA commitu.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

REPOSITORY = "000307/Odchylky"
REF = "main"
script_names = [
    "predikce_bayes_ar.py",
    "bayes_ar_core.py",
    "predikce_linearni_regrese.py",
]
for name in script_names:
    destination = Path(name)
    urlretrieve(
        f"https://raw.githubusercontent.com/{REPOSITORY}/{REF}/{name}",
        destination,
    )
    compile(destination.read_text(encoding="utf-8"), str(destination), "exec")
    print(f"Stažen úplný skript: {destination} ({destination.stat().st_size} bajtů)")
script = Path("predikce_bayes_ar.py")


## 3. Nahrát všechna vstupní CSV

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_files = sorted(name for name in uploaded if name.lower().endswith(".csv"))
if not csv_files:
    raise ValueError("Nahrajte vstupní CSV s minutovými odchylkami.")
print(f"Vybráno CSV souborů: {len(csv_files)}")


## 4. Nastavit MCMC a spustit časové vyhodnocení

První tři dostupné dny slouží k počátečnímu učení. Každý další den se testuje
zvlášť na modelu naučeném jen ze starších dnů.

Výchozí kontrola požaduje R-hat ≤ 1,01 a bulk i tail ESS ≥ 400. Pokud kontrola
neprojde, skript se zastaví. Prozkoumejte `diagnostika_mcmc.csv` a případně
zvyšte `draws` nebo `tune`. Neměňte model podle toho, která varianta náhodou
vyjde nejlépe na tomto testovacím období.


In [ ]:
import os
import pandas as pd
from IPython.display import Image, display

MCMC = {"lags": 5, "chains": 4, "tune": 1000, "draws": 2000, "seed": 20261102}
mcmc_args = [item for key, value in MCMC.items() for item in (f"--{key}", str(value))]
out_dir = Path("predikce_bayes")
subprocess.run(
    [sys.executable, str(script), *csv_files, *mcmc_args, "--out", str(out_dir)],
    check=True,
    env={**os.environ, "MPLBACKEND": "Agg"},
)
print((out_dir / "vysledky.txt").read_text(encoding="utf-8"))


## 5. Zobrazit metriky, pravděpodobnosti a diagnostiku

Správnost jednotlivých minut, všech deseti znamének současně a znaménka
průměru dalších deseti minut jsou různé ukazatele. U Brierova skóre a
logaritmické ztráty jsou lepší nižší hodnoty. Přesné nulové skutečnosti se
vynechávají z binárního pravděpodobnostního skórování.

90% predikční interval popisuje rozptyl budoucích hodnot za předpokladů modelu.
Jeho skutečné pokrytí i kalibrace pravděpodobností se ověřují na testovacích datech.


In [ ]:
for file_name in [
    "metriky.csv", "metriky_po_horizontech.csv", "kalibrace.csv", "diagnostika_mcmc.csv"
]:
    print(f"\n{file_name}")
    display(pd.read_csv(out_dir / file_name))

for file_name in ["predikce_polarity.png", "mcmc_stopy.png"]:
    display(Image(filename=str(out_dir / file_name)))


## 6. Stáhnout výsledky

ZIP obsahuje tabulky, grafy, shrnutí a uložené MCMC tahy. Uchovejte jej pro
pozdější kontrolu. Výsledky neprokazují, že model překoná poslední známé
znaménko na jiné, delší historii.


In [ ]:
from shutil import make_archive

archive = make_archive("vysledky_bayes_ar_mcmc", "zip", root_dir=out_dir)
files.download(archive)


## Volitelně: předpověď nového intervalu

Po dokončení předchozích kroků a získání prvních pěti minut nahraďte ilustrační
začátek a všech pět hodnot skutečnými údaji. Nastavte `SPUSTIT_PREDPOVED = True`
a spusťte buňku. K učení se použijí pouze historické intervaly dokončené
nejpozději na začátku předpovídané čtvrthodiny. Nastavení MCMC je stejné jako
v kroku 4; měření daného intervalu po páté minutě do něj nevstupují.


In [ ]:
SPUSTIT_PREDPOVED = False
START = "2026-09-29 12:00"
PRVNICH_PET_MW = [10, 20, 30, 40, 50]  # Nahraďte skutečnými hodnotami.

if SPUSTIT_PREDPOVED:
    forecast_dir = Path("nova_predikce_bayes")
    subprocess.run(
        [sys.executable, str(script), *csv_files, *mcmc_args,
         "--first-five", *map(str, PRVNICH_PET_MW),
         "--start", START, "--out", str(forecast_dir)],
        check=True,
        env={**os.environ, "MPLBACKEND": "Agg"},
    )
    display(pd.read_csv(forecast_dir / "predikce_noveho_intervalu.csv"))
    display(pd.read_csv(forecast_dir / "souhrn_predikce.csv"))
    forecast_archive = make_archive("nova_predikce_bayes_ar_mcmc", "zip", root_dir=forecast_dir)
    files.download(forecast_archive)
